# Cell 1: Import everything:


In [1]:
import torch
import numpy as np

from sklearn.model_selection import train_test_split
from transformers import BertModel, BertConfig
from torch.utils.data import DataLoader

# Data handeling
from src.data.dataset import MLMDataset, ClassificationDataset
from src.data.data_tools import filter_taxonomy, fasta2pandas

#Vocab shit
from src.utils.vocab import Vocabulary, KmerVocabConstructor

# Preprocessing
from src.preprocessing.augmentation import SequenceModifier, IdentityStrategy, BaseStrategy
from src.preprocessing.tokenization import KmerStrategy
from src.preprocessing.padding import PEndStrategy
from src.preprocessing.truncation import TEndStrategy
from src.preprocessing.preprocessor import OverlappingPreprocessor

# Model things
from src.model.backbone import Bertax, ModularBertax
from src.model.encoders import LabelEncoder
from src.model.heads import MLMHead, SingleClassHead

# Training things
from src.train.trainers import MLMtrainer, ClassificationTrainer

In [2]:


CONFIG = {
    "FILE_PATH": "src/data/raw.fasta",
    "SAVE_PATH": "pretrained_model.pt",
    "n_test": 10,
    "modification_probability": 0.05,
    "alphabet": ["A", "C", "G", "T"],
    "k": 3,
    "optimal_length": 200,

    # Training parameters
    "num_epochs": 50,
    "masking_percentage": 0.05,
    "batch_size": 128,
    "small_set": False,

    # Model configuration
    "num_layers": 10,
    "num_attention_heads": 4,
    "hidden_size": 256,
    "intermediate_size": 1024,  # 4 * hidden_size
    "dropout_rate": 0.05,
    "num_classes": 19,
    "mlm_dropout_rate": 0.1,

    #Classification 
    "target_label": "phylum"
}


In [3]:

# Set up vocabulary
constructor = KmerVocabConstructor(k=CONFIG["k"], alphabet=CONFIG["alphabet"])
vocab = Vocabulary()
vocab.build_from_constructor(constructor, data=[])
vocab_path = "vocab.json"
vocab.save(vocab_path)

# Set up preprocessors
sequence_modifier = SequenceModifier(alphabet=CONFIG["alphabet"])
augmentation_strategy_train = BaseStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=CONFIG["modification_probability"]
)

augmentation_strategy_val = IdentityStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=0
)

tokenization_strategy = KmerStrategy(
    k=CONFIG["k"],
    padding_alphabet=CONFIG["alphabet"]
)

padding_strategy = PEndStrategy(
    optimal_length=CONFIG["optimal_length"]*CONFIG["k"]
)

truncation_strategy = TEndStrategy(
    optimal_length=CONFIG["optimal_length"]*CONFIG["k"]
)

preprocessor_train = OverlappingPreprocessor(
    k=CONFIG["k"],
    augmentation_strategy=augmentation_strategy_train,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)

preprocessor_val = OverlappingPreprocessor(
    k=CONFIG["k"],
    augmentation_strategy=augmentation_strategy_val,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)



In [4]:

##################################################################
## Data preparation ##############################################
##################################################################

all_data = fasta2pandas(CONFIG["FILE_PATH"])

#tmp: to use a smaller set for test if code compiles - nothing to be used in production
if CONFIG["small_set"]:
    all_data = all_data[:CONFIG["n_test"]]

# filter data for finetuning
filtered_data = filter_taxonomy(
    df = all_data,
    startAt ='phylum',
    endAt = 'phylum',
    phylumCertainty=True
    )

# 
num_classes = len(list(set(filtered_data[CONFIG["target_label"]])))

# encodes label for classification
label_encoder = LabelEncoder(filtered_data[CONFIG["target_label"]])

# Pretraining datasplit based on unsplit data: 
pretrain_sequences, preval_sequences = train_test_split(
    all_data["sequence"],
    test_size = 0.1,
    random_state = 42
)

# Finetune datasplit based on filtered data:
finetrain_data, fineval_data = train_test_split(
    filtered_data,
    test_size = 0.1,
    random_state = 69
    )

print(f"Number of pre-training sequences: {len(pretrain_sequences)}")
print(f"Number of validation sequences: {len(preval_sequences)}")


Applying filters...
Filtering complete.

Handling DNA ambiguity codes...
Processing row 0...
Processing row 10000...
Processing row 20000...
Processing row 30000...
Processing row 40000...
Processing row 50000...
Processing row 60000...
Processing row 70000...
Processing row 80000...
Processing row 90000...
Processing complete.
Number of pre-training sequences: 83962
Number of validation sequences: 9330


In [5]:
# Only for testing, to be removed later...
test_sentence = pretrain_sequences[0]
print(test_sentence)
print(len(test_sentence)/CONFIG["k"])
processed_test_sentence = preprocessor_train.process(test_sentence)
print(processed_test_sentence)
print(len(processed_test_sentence))

AAGGATCATTATCGAGTGGGGGTCCTCTGGGCCCCGTCTCCAACCCTTGTCTATTCTACCATGTTGCTTTGGCGGGCCCGTCTGCAACCGGACCGCTGGGGACTCGCGCCCCTGGCCCGCGCCCGTCAATAGCCCCCCCAACTTTTTCTAACAGTGACGTCTAAGCAAACGAGAATAACCAAAACTTTCAACAACGGATCTCTTGGTTCTGGCATCGATGAAGAACGCAGCGAAATGCGATAAGTAATGCGAATTGCAGAATTCCGTGAGTCATCGAATCTTTGAACGCACATTGCGCCCTCTGGTACTCCGGAGGGCATGCCTGTTCGAGCGTCATTGTCAACCGTCAAGCTCGGCTTGCTGTTGGGTCCCCGTCGTTGCTCCAGCGGCGGACCCGAAAGATAATGGCAGAGTCTGTGAGACCCTGGATGCAGCGAGCTTCTAGCACGCGTCTGGACGGTCTTTAGGCTTGGTCTCAACCAGTTGTCAACTTCTG
165.33333333333334
[7, 40, 24, 56, 29, 51, 39, 31, 58, 60, 47, 26, 32, 34, 21, 26, 67, 60, 20, 33, 25, 64, 62, 68, 46, 47, 26, 50, 62, 6, 31, 10, 44, 47, 12, 30, 42, 28, 46, 27, 30, 27, 57, 17, 42, 26, 25, 20, 20, 33, 9, 51, 11, 57, 14, 5, 29, 37, 53, 25, 5, 36, 9, 9, 11, 40, 34, 67, 52, 35, 41, 59, 19, 13, 6, 41, 43, 5, 62, 40, 7, 53, 62, 37, 67, 23, 8, 58, 51, 16, 24, 29, 18, 68, 37, 30, 9, 68, 30, 25, 66, 48, 12, 27, 39, 46, 19, 28, 52, 29, 43, 57, 67, 57, 10, 50, 7, 34, 46, 67, 35, 67, 48, 26, 

In [6]:
##################################################################
## Setup Datasets ################################################
##################################################################

# Pretraining datasets
pretrain_dataset = MLMDataset(
    df = pretrain_sequences,
    preprocessor = preprocessor_train,
    masking_percentage = CONFIG["masking_percentage"]
)

preval_dataset = MLMDataset(
    df = preval_sequences,
    preprocessor = preprocessor_val,
    masking_percentage = CONFIG["masking_percentage"]
)

# Finetuning datasets
finetrain_dataset = ClassificationDataset(
    df = finetrain_data,
    preprocessor = preprocessor_train,
    label_encoder = label_encoder,
    target_column = "phylum"
    )

fineval_dataset = ClassificationDataset(
    df = fineval_data,
    preprocessor = preprocessor_val,
    label_encoder = label_encoder,
    target_column = "phylum"
    )

##################################################################
## Setup Dataloader ##############################################
##################################################################

pretrain_loader = DataLoader(
    dataset = pretrain_dataset,
    batch_size = CONFIG["batch_size"],
    shuffle = True,
    num_workers = 15
)

preval_loader = DataLoader(
    dataset = preval_dataset,
    batch_size = CONFIG["batch_size"],
    shuffle = True,
    num_workers = 15
)

#data loaders
finetrain_loader = DataLoader(
    dataset=finetrain_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers = 15
    )

fineval_loader = DataLoader(
    dataset=fineval_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers = 15
    )



In [7]:
encoder_config = BertConfig(
    vocab_size = len(vocab),
    hidden_size = CONFIG["hidden_size"],
    num_hidden_layers = CONFIG["num_layers"],
    num_attention_heads = CONFIG["num_attention_heads"],
    intermediate_size = CONFIG["intermediate_size"],
    max_position_embeddings = CONFIG["optimal_length"]*CONFIG["k"] + CONFIG["k"] + 1, # k-SEPS's and one CLS token
    hidden_dropout_prob = CONFIG["dropout_rate"],
    attention_probs_dropout_prob = CONFIG["dropout_rate"]
)

encoder = BertModel(encoder_config)

mlm_head = MLMHead(
    in_features = CONFIG["hidden_size"],
    hidden_layer_size = CONFIG["hidden_size"] // 2,
    out_features = len(vocab),
    dropout_rate = CONFIG["mlm_dropout_rate"]
)

classification_head = SingleClassHead(
    in_features = CONFIG["hidden_size"],
    hidden_layer_size = CONFIG["hidden_size"] // 2,
    out_features = num_classes,
    dropout_rate = CONFIG["mlm_dropout_rate"]
)

model = ModularBertax(
    encoder = encoder,
    mlm_head = mlm_head,
    classification_head = classification_head
)

mlm_trainer = MLMtrainer(
    model = model,
    train_loader = pretrain_loader,
    val_loader = preval_loader
)

classification_trainer = ClassificationTrainer(
    model,
    train_loader = finetrain_loader,
    val_loader = fineval_loader
    )

Training on device cuda and it is awesome!!!
Training on device cuda and it is awesome!!!


In [8]:
model.preTrainMode()
mlm_trainer.train(CONFIG["num_epochs"])
torch.save(model.state_dict(), CONFIG["SAVE_PATH"])
print(f"Model's state dictionary saved to {CONFIG['SAVE_PATH']}")

model.classifyMode()
classification_trainer.train(CONFIG["num_epochs"])

Epoch 1/50
Train Avg Loss: 4.1174, Train MLM Accuracy: 0.0315
Val Avg Loss: 4.0603, Val MLM Accuracy: 0.0400


Epoch 2/50
Train Avg Loss: 4.0741, Train MLM Accuracy: 0.0376
Val Avg Loss: 4.0370, Val MLM Accuracy: 0.0426


Epoch 3/50
Train Avg Loss: 4.0200, Train MLM Accuracy: 0.0443
Val Avg Loss: 3.9241, Val MLM Accuracy: 0.0544


Epoch 4/50
Train Avg Loss: 3.9476, Train MLM Accuracy: 0.0512
Val Avg Loss: 3.8375, Val MLM Accuracy: 0.0631


Epoch 5/50
Train Avg Loss: 3.8910, Train MLM Accuracy: 0.0566
Val Avg Loss: 3.7661, Val MLM Accuracy: 0.0705


Epoch 6/50
Train Avg Loss: 3.8474, Train MLM Accuracy: 0.0606
Val Avg Loss: 3.7127, Val MLM Accuracy: 0.0748


Training Epoch 7:   0%|          | 0/656 [00:00<?, ?it/s]

In [ ]:
import torch
print(torch.cuda.memory_summary(device=torch.device('cuda')))


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 2            |        cudaMalloc retries: 2         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  14648 MiB |  14649 MiB |  28568 MiB |  13919 MiB |
|       from large pool |  14606 MiB |  14606 MiB |  28491 MiB |  13884 MiB |
|       from small pool |     41 MiB |     42 MiB |     77 MiB |     35 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  14648 MiB |  14649 MiB |  28568 MiB |  13919 MiB |
|       from large pool |  14606 MiB |  14606 MiB |  28491 MiB |